In [1]:
import numpy as np
import pandas as pd
import pyfastx
from utils import parse_vadr_cds_tbl, format_cds_coordinates
import matplotlib.pyplot as plt
from collections import defaultdict
import matplotlib.patches as mpatches
from viz_sequence_v2 import plot_dashboard_grid
import re
import os
from pathlib import Path
from scipy.spatial.distance import cdist, cosine
from sklearn.preprocessing import normalize

In [2]:
df_01_PB2 = pd.read_csv('H5Nx_NorthAmerica_Metadata/01_PB2/H5Nx_1_PB2_ID_Filtered.csv')

In [3]:
df_01_PB2.shape

(10382, 42)

In [4]:
df_01_PB2.index

RangeIndex(start=0, stop=10382, step=1)

In [5]:
stop_codons = {'TAA', 'TAG', 'TGA'}

In [6]:
fa_01_PB2 = pyfastx.Fasta('H5Nx_NorthAmerica_Sequences/01_PB2_H5Nx_North_America.fasta', build_index=True)

In [7]:
cds_tbl_path = f'H5Nx_NorthAmerica_VADR/01_PB2_VADR/01_PB2_VADR.vadr.pass.tbl'
parsed_cds_tbl_data = parse_vadr_cds_tbl(cds_tbl_path)
cds_coords_dict = format_cds_coordinates(parsed_cds_tbl_data)

In [8]:
def batch_cosine_dist(X_query, ref_vector, batch_size=2000):
    """
    Computes cosine distance between many query vectors and a single reference vector.
    Optimization: Uses dot product of normalized vectors instead of cdist.
    
    Args:
        X_query (np.ndarray): Shape (N, features)
        ref_vector (np.ndarray): Shape (features,) or (1, features)
        batch_size (int): Size of chunks (less critical here but good for consistency)
        
    Returns:
        np.ndarray: Array of cosine distances of shape (N,)
    """
    ref_vector = ref_vector.reshape(1, -1)
    
    ref_norm = normalize(ref_vector, norm='l2', axis=1)

    X_query_norm = normalize(X_query, norm='l2', axis=1)
    
    similarities = X_query_norm.dot(ref_norm.T).flatten()
    
    similarities = np.clip(similarities, -1.0, 1.0)
    distances = 1.0 - similarities
    
    return distances

In [9]:
def get_codon(sequence, cds_coordinates, codon_pos, codon_start=1):
    """Count codons in the given coding sequence"""
    full_cds_sequence = ""

    # Step 1: Correctly parse ALL start and end coordinates from the string.
    coords = re.findall(r'[<>]?(\d+)\.\.[<>]?(\d+)', cds_coordinates)
    if not coords:
        raise ValueError(f"Could not parse CDS coordinates from string: {cds_coordinates}")

    # Step 2: Build the complete CDS by concatenating all exon parts.
    for start, end in coords:
        start_index, end_index = int(start) - 1, int(end)
        full_cds_sequence += sequence[start_index:end_index]

    # Step 3: Apply the codon_start offset to the ASSEMBLED sequence.
    offset = int(codon_start) - 1
    trimmed_sequence = full_cds_sequence[offset:]

    # Step 4: Process the final sequence
    remainder = len(trimmed_sequence) % 3
    seq_length = len(trimmed_sequence) - remainder

    for pos, codon_idx in enumerate(range(0, seq_length, 3)):
        codon = trimmed_sequence[codon_idx:codon_idx+3]
        if pos+1 == codon_pos:
            return codon
    return ''


In [10]:
df_01_PB2['Codon_Pos_627'] = ''
df_01_PB2['Codon_Pos_701'] = ''

In [11]:
codon_pos_627_dict = {}
codon_pos_701_dict = {}
for index, row in df_01_PB2.iterrows():
    seq_id = row['1_PB2_ID']
    sequence=fa_01_PB2[seq_id].seq.upper()
    cds_coords = cds_coords_dict[seq_id]['polymerase PB2']
    extracted_coords = cds_coords['coordinate']
    codon_start = cds_coords.get('codon_start', 1)
    # if int(codon_start) != 1:
    #     print (seq_id, codon_start, row['Isolate_Name'], codon_start)
    codon_627 = get_codon(sequence, extracted_coords, 627, codon_start)
    codon_pos_627_dict[seq_id] = codon_627
    
    codon_701 = get_codon(sequence, extracted_coords, 701, codon_start)
    codon_pos_701_dict[seq_id] = codon_701

In [12]:
df_01_PB2['Codon_Pos_627'] = df_01_PB2['1_PB2_ID'].apply(lambda x: codon_pos_627_dict[x])
df_01_PB2['Codon_Pos_701'] = df_01_PB2['1_PB2_ID'].apply(lambda x: codon_pos_701_dict[x])

In [13]:
df_01_PB2['Host_ID'].value_counts(dropna=False)

Host_ID
1    8451
2    1903
0      28
Name: count, dtype: int64

In [14]:
matching_index = df_01_PB2[df_01_PB2['Codon_Pos_627'].isin(['AAA', 'AAG'])].index

In [15]:
len(matching_index)

188

In [16]:
df_01_PB2_627 = df_01_PB2.loc[matching_index]

In [17]:
df_01_PB2_627.index

Index([  164,   175,   184,   216,   226,   234,   236,   243,   244,   245,
       ...
        9549,  9556,  9560,  9652,  9654,  9657, 10027, 10117, 10260, 10274],
      dtype='int64', length=188)

In [18]:
df_01_PB2_627 = df_01_PB2_627.reset_index(drop=True)

In [19]:
df_01_PB2_627.index

RangeIndex(start=0, stop=188, step=1)

In [20]:
df_01_PB2_627

,Unnamed: 0.2,Unnamed: 0.1,index,Unnamed: 0,Isolate_Id,PB2 Segment_Id,PB1 Segment_Id,PA Segment_Id,HA Segment_Id,NP Segment_Id,...,6_NA_ID,7_MP_ID,8_NS_ID,Host_ID,deduplication,Ambiguous_Count,Seq_Len,VADR_Screening_Passed,Codon_Pos_627,Codon_Pos_701
0,164,171,171,289,EPI_ISL_19825669,EPI4244095,EPI4244093,EPI4244090,EPI4244091,EPI4244092,...,EPI4244089,EPI4244096,EPI4244094,2,keep_earliest,1,2280,True,AAA,GAC
1,175,183,183,304,EPI_ISL_19825738,EPI4244578,EPI4244576,EPI4244573,EPI4244574,EPI4244575,...,EPI4244572,EPI4244579,EPI4244577,1,no_dup,1,2280,True,AAA,GAC
2,184,192,192,329,EPI_ISL_19891037,EPI4404717,EPI4404715,EPI4404712,EPI4404713,EPI4404714,...,EPI4404711,EPI4404718,EPI4404716,1,no_dup,1,2280,True,AAA,GAC
3,216,227,229,378,EPI_ISL_20055074,EPI4467389,EPI4467387,EPI4467384,EPI4467385,EPI4467386,...,EPI4467383,EPI4467390,EPI4467388,2,keep_earliest,1,2280,True,AAA,GAC
4,226,239,241,395,EPI_ISL_19792274,EPI4148724,EPI4148722,EPI4148719,EPI4148720,EPI4148721,...,EPI4148718,EPI4148725,EPI4148723,2,no_dup,1,2280,True,AAA,GAC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,9657,9904,9946,18045,EPI_ISL_19820562,EPI4226771,EPI4226766,EPI4226757,EPI4226762,EPI4226764,...,EPI4226755,EPI4226773,EPI4226768,2,no_dup,0,2280,True,AAA,GAC
184,10027,10285,10330,18857,EPI_ISL_19820750,EPI4228144,EPI4228139,EPI4228133,EPI4228135,EPI4228136,...,EPI4228131,EPI4228145,EPI4228142,1,no_dup,1,2280,True,AAA,GAC
185,10117,10375,10420,18993,EPI_ISL_19820684,EPI4227661,EPI4227659,EPI4227656,EPI4227657,EPI4227658,...,EPI4227655,EPI4227662,EPI4227660,1,no_dup,1,2280,True,AAA,GAC
186,10260,10522,10567,19396,EPI_ISL_19593964,EPI3678204,EPI3678202,EPI3678199,EPI3678200,EPI3678201,...,EPI3678198,EPI3678205,EPI3678203,2,no_dup,162,2280,True,AAG,GTG


In [21]:
df_01_PB2_627[df_01_PB2_627['Codon_Pos_627']=='AAG']

,Unnamed: 0.2,Unnamed: 0.1,index,Unnamed: 0,Isolate_Id,PB2 Segment_Id,PB1 Segment_Id,PA Segment_Id,HA Segment_Id,NP Segment_Id,...,6_NA_ID,7_MP_ID,8_NS_ID,Host_ID,deduplication,Ambiguous_Count,Seq_Len,VADR_Screening_Passed,Codon_Pos_627,Codon_Pos_701
5,234,247,249,407,EPI_ISL_19792291,EPI4148850,EPI4148848,EPI4148845,EPI4148846,EPI4148847,...,EPI4148844,EPI4148851,EPI4148849,1,no_dup,179,2280,True,AAG,GTG
20,499,546,551,947,EPI_ISL_19566638,EPI3661106,EPI3661107,EPI3661105,EPI3661109,EPI3661102,...,EPI3661108,EPI3661104,EPI3661103,1,no_dup,0,2316,True,AAG,GAC
186,10260,10522,10567,19396,EPI_ISL_19593964,EPI3678204,EPI3678202,EPI3678199,EPI3678200,EPI3678201,...,EPI3678198,EPI3678205,EPI3678203,2,no_dup,162,2280,True,AAG,GTG
187,10274,10540,10585,19430,EPI_ISL_19593929,EPI3677948,EPI3677946,NaN,EPI3677944,EPI3677945,...,EPI3677943,EPI3677949,EPI3677947,2,no_dup,128,2280,True,AAG,GTG


In [22]:
def get_cds_shap(sequence, shap_values, cds_coordinates, codon_start=1):
    """Count codons in the given coding sequence"""
    full_cds_sequence = ""

    # Step 1: Correctly parse ALL start and end coordinates from the string.
    coords = re.findall(r'[<>]?(\d+)\.\.[<>]?(\d+)', cds_coordinates)
    if not coords:
        raise ValueError(f"Could not parse CDS coordinates from string: {cds_coordinates}")

    # Step 2: Build the complete CDS by concatenating all exon parts.
    #print ('Extracted coords', coords)
    for start, end in coords:
        start_index, end_index = int(start) - 1, int(end)
        full_cds_sequence += sequence[start_index:end_index]
    shap_parts = [shap_values[:, int(start)-1:int(end)] for start, end in coords]

    full_shap_values = np.concatenate(shap_parts, axis=1)
    # Step 3: Apply the codon_start offset to the ASSEMBLED sequence.
    offset = int(codon_start) - 1
    trimmed_sequence = full_cds_sequence[offset:]
    trimmed_shap    = full_shap_values[:, offset:]

    # Step 4: Process the final sequence
    remainder = len(trimmed_sequence) % 3
    seq_length = len(trimmed_sequence) - remainder
    
    return trimmed_sequence[:seq_length], trimmed_shap[:, :seq_length]

In [23]:
x_onehot_pb2 = np.load('H5Nx_NorthAmerica_Shap/01_PB2/X_onehot_627_AAA_AAG.npy')

In [24]:
x_onehot_pb2.shape

(188, 5, 2400)

In [25]:
def is_at_rich(codon):
    # Define as A/T-rich if majority of bases are A or T
    return sum(1 for b in codon if b in 'AT') >= 2

In [26]:
def onehot_to_dna(onehot):
    
    alphabet = np.array(['A', 'C', 'G', 'T', 'N'])
    mask = onehot.any(axis=0)
    indices = np.argmax(onehot[:, mask], axis=0)
    dna_seq = "".join(alphabet[indices])
    
    return dna_seq

In [27]:
def dict_allclose(d1, d2, rtol=1e-8, atol=1e-12):
    if d1.keys() != d2.keys():
        return False
    for k in d1:
        if not np.isclose(d1[k], d2[k], rtol=rtol, atol=atol):
            print("mismatch key:", k, d1[k], d2[k])  # optional debug
            return False
    return True

In [28]:

supplementary_data = []

# Constants for E627K (Codon 627)
TARGET_CODON_NUM = 627
# 1-based codon -> 0-based nucleotide index
# (627 - 1) * 3 = 1878
START_NUC_IDX = (TARGET_CODON_NUM - 1) * 3  

results = "/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results"
if not os.path.exists(results):
    os.makedirs(results)

print(f"Starting analysis...")
codon_list = np.load('codon_list.npy')
avi_medoid = np.load('01_PB2_avi_medoid_pre2020.npy')
dists = []
for index, row in df_01_PB2_627.iterrows():
    seq_id = row['1_PB2_ID']
    
    # FILTER: Human + E627K
    n_ambiguous_bases = row['Ambiguous_Count']
    if n_ambiguous_bases <=10 and row['Codon_Pos_627']=='AAA' and row['Host_Group']=='Nonhuman Mammal': 
        
        seq_id = row['1_PB2_ID']
        isolate_name = row['Isolate_Name']
        sequence=fa_01_PB2[seq_id].seq.upper()
        clean_seq = re.sub(r'[^ACGT]', 'N', sequence.upper())
        ## just check the data to make sure it is a correct sequence
        assert clean_seq == onehot_to_dna(x_onehot_pb2[index])
        
        cds_coords = cds_coords_dict[seq_id]['polymerase PB2']
        extracted_coords = cds_coords['coordinate']
        codon_start = cds_coords.get('codon_start', 1)
        print (seq_id, cds_coords, codon_start)

        shap_scores_arr = []

        avg_codon_score = defaultdict(list)
        for fold in range(10):
            shap_scores_tmp = np.load(f'H5Nx_NorthAmerica_Shap/01_PB2/Shap_627_AAA_AAG_Fold_{fold}.npy')
            assert shap_scores_tmp.shape[0] == x_onehot_pb2.shape[0]
            
            seq_scores_tmp = shap_scores_tmp[index, :, :, 2] * x_onehot_pb2[index, :, :]
            trimmed_seq, trimmed_shap = get_cds_shap(sequence, seq_scores_tmp, extracted_coords, codon_start)
            shap_scores_arr.append(trimmed_shap)
            
            codon_score = {}
            for pos, codon_idx in enumerate(range(0, len(trimmed_seq), 3)):
                codon = trimmed_seq[codon_idx:codon_idx+3]
                if 'N' not in codon and codon not in stop_codons:
                    codon_shap_score = np.sum(trimmed_shap[:, codon_idx:codon_idx+3])
                    codon_score[codon] = codon_score.get(codon,0.0) + codon_shap_score
            for key, value in codon_score.items():
                avg_codon_score[key].append(value)
        bar_data = {}
        for codon in codon_list:
            if codon not in avg_codon_score:
                bar_data[codon] = 0.0
            else:
                bar_data[codon] = np.mean(np.asarray(avg_codon_score[codon]))       
        phenotypic_distance = batch_cosine_dist(np.array(list(bar_data.values()), dtype=np.float32).reshape(1, -1), avi_medoid)
        print (phenotypic_distance[0])
        dists.append(phenotypic_distance[0])

        #get mean for sequence logo plot
        stack_shap_score = np.stack(shap_scores_arr)
        mean_shap_score = np.mean(stack_shap_score, axis=0)  

        ########################Check 2 way of everage is equivaltent########################
        codon_score_check = {}
        for pos, codon_idx in enumerate(range(0, len(trimmed_seq), 3)):
            codon = trimmed_seq[codon_idx:codon_idx+3]
            if 'N' not in codon and codon not in stop_codons:
                codon_shap_score_check = np.sum(mean_shap_score[:, codon_idx:codon_idx+3])
                codon_score_check[codon] = codon_score_check.get(codon,0.0) + codon_shap_score_check
        bar_data_check = {}
        for codon in codon_list:
            if codon not in codon_score_check:
                bar_data_check[codon] = 0.0
            else:
                bar_data_check[codon] = codon_score_check[codon]
        phenotypic_distance_check = cdist(np.array(list(bar_data_check.values())).reshape(1, -1), avi_medoid.reshape(1, -1), metric='cosine').flatten()
        assert dict_allclose(bar_data, bar_data_check, rtol=1e-8, atol=1e-12)
        print (phenotypic_distance_check[0])
        ###########################################################################################
        
        seq_scores = mean_shap_score[0:4, :]
        
        # --- B. Get E627K Specific Scores (Positions 1, 2, 3) ---
        # START_NUC_IDX = 1878
        score_pos1 = np.sum(seq_scores[:, START_NUC_IDX])     # 1878 (A)
        score_pos2 = np.sum(seq_scores[:, START_NUC_IDX + 1]) # 1879 (A)
        score_pos3 = np.sum(seq_scores[:, START_NUC_IDX + 2]) # 1880 (G)
        
        total_627_score = score_pos1 + score_pos2 + score_pos3
        actual_codon_seq = trimmed_seq[START_NUC_IDX : START_NUC_IDX + 3] # Should be AAG

        # --- C. Get Top 20 Positive Nucleotides Globally ---
        # Sum axis 0 -> 1D array of scores per position
        per_nuc_scores = np.sum(seq_scores, axis=0) 
        
        positive_sites = []
        for i, score in enumerate(per_nuc_scores):
            if score > 0:
                # Store: (Position 1-based, Score, Nucleotide)
                positive_sites.append((i + 1, score, trimmed_seq[i]))
        
        # Sort descending
        positive_sites.sort(key=lambda x: x[1], reverse=True)
        top_20 = positive_sites[:20]
        
        # Format string: "Pos 1880(G): 0.52; Pos 120(A): 0.45"
        top_20_str = "; ".join([f"Pos {p}({n}): {s:.4f}" for p, s, n in top_20])


        # --- D. Build CSV Record ---
        record = {
            'Sequence_ID': seq_id,
            'Isolate_Name': row['Isolate_Name'],
            'Host': row['Host_Group'],
            'Mutation': 'E627K',
            'Host-adaptive Distance': round(phenotypic_distance[0], 4),
            'Codon_Seq': actual_codon_seq,
            
            # 1. Specific Breakdown for 627
            'SHAP_Total_627': round(total_627_score, 4),
            'SHAP_627_Pos1_A': round(score_pos1, 4),
            'SHAP_627_Pos2_A': round(score_pos2, 4),
            'SHAP_627_Pos3_A': round(score_pos3, 4),
            
            # 2. Global Context
            'Top_20_Positive_Sites': top_20_str
        }
        supplementary_data.append(record)


        # --- E. Plotting ---
        # 1. Codon scores for scatter
        codon_shap_by_pos = defaultdict(float)
        
        for pos, codon_idx in enumerate(range(0, len(trimmed_seq), 3)):
            codon = trimmed_seq[codon_idx:codon_idx+3]
            if codon =='AAA' or codon=='AAG':
                codon_shap_score = np.sum(seq_scores[:, codon_idx:codon_idx+3])
                codon_shap_by_pos[pos+1] = codon_shap_score

        # 2. Logo Data: Transpose (4, L) -> (L, 4)
        seq_logo_data = np.transpose(seq_scores) 
        
        codon_bar_data = bar_data
        plot_dashboard_grid(
            scatter_data=codon_shap_by_pos,
            logo_array=seq_logo_data,
            bar_data=codon_bar_data,
            top_pos_data=top_20,
            codon=['Lysine K', 'AAA', 'AAG'],
            fig_title=f"{seq_id} - {row['Isolate_Name']} - H5Nx North America",
            scatter_markers=[{'pos': 627, 'label': '627K', 'color': 'dodgerblue'}],
            logo_region=(1850, 1910),
            logo_highlight={'red': [(START_NUC_IDX, START_NUC_IDX + 3, "627K")]},
            subticks_frequency=5,
            is_at_rich_fn=is_at_rich,
            save_path=f"{results}/Mutation/H5Nx_NorthAmerica/E627K_{seq_id}_H5Nx_NorthAmerica.png"
        )

# ==========================================
# 3. SAVE TO CSV
# ==========================================
df_supp = pd.DataFrame(supplementary_data)
# Sort by the 3rd position (G) score, or Total 627 score? Usually Total is best.
df_supp = df_supp.sort_values(by='SHAP_Total_627', ascending=False)

csv_name = f"{results}/Supplementary_Table_E627K_H5Nx.csv"
df_supp.to_csv(csv_name, index=False)
print(f"Mean Phenotypic Distance: {np.mean(dists)}")
print(f"Std Dev: {np.std(dists)}")
print(f"Done. Saved table to {csv_name}")

Starting analysis...
EPI4244095 {'coordinate': '1..2280'} 1
1.7448719
1.7448717665335733
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/E627K_EPI4244095_H5Nx_NorthAmerica.png
EPI4467389 {'coordinate': '1..2280'} 1
1.7209511
1.7209511120980334
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/E627K_EPI4467389_H5Nx_NorthAmerica.png
EPI4148724 {'coordinate': '1..2280'} 1
1.7120361
1.7120361362231127
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/E627K_EPI4148724_H5Nx_NorthAmerica.png
EPI4149152 {'coordinate': '1..2280'} 1
1.6761069
1.6761069226303404
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/E627K_EPI4149152_H5Nx_NorthAmerica.png
EPI4149144 {'coordinate': '1..2280'} 1
